# Desocupação — merge, melt e gráfico

As Tabelas 5 trazem a composição por sexo das pessoas desocupadas, em percentual. A coluna das mulheres é a participação delas entre as pessoas desocupadas. Não é a taxa de desocupação das mulheres.

O cruzamento com as horas de afazeres mostra os dois números no mesmo estado. Ele não indica causa.

Execute a partir da pasta da atividade ou da raiz de `analise_dados`.


In [1]:
import collections
import collections.abc

for nome in ("Iterable", "Mapping", "Sequence"):
    if not hasattr(collections, nome):
        setattr(collections, nome, getattr(collections.abc, nome))

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import xlrd

def ler_xls(caminho, aba):
    # xlrd 1.2 abre .xls. O pandas recente pede xlrd 2, que não lê esse formato.
    livro = xlrd.open_workbook(str(caminho))
    planilha = livro.sheet_by_name(aba)
    linhas = [planilha.row_values(i) for i in range(planilha.nrows)]
    return pd.DataFrame(linhas)

def raiz_do_repositorio():
    aqui = Path.cwd().resolve()
    for pasta in [aqui, *aqui.parents]:
        if (pasta / "dados").is_dir() and (pasta / "etapa_3").is_dir():
            return pasta
    raise FileNotFoundError("Execute a partir do repositorio analise_dados.")

RAIZ = raiz_do_repositorio()
DADOS = RAIZ / "dados"
SAIDAS = RAIZ / "etapa_3" / "06_desocupacao_merge" / "saidas"
SAIDAS.mkdir(parents=True, exist_ok=True)

t2012 = pd.read_csv(DADOS / "Tabela5-sem_emprego_2012.csv", sep=";", decimal=",", encoding="utf-8-sig")
t2026 = pd.read_csv(DADOS / "Tabela5-sem_emprego_2026.csv", sep=";", decimal=",", encoding="utf-8-sig")
print(t2012.columns.tolist())
print(t2026.columns.tolist())
print(len(t2012), len(t2026))


['Sigla', 'Código', 'Estado', 'Desocupados - homens (2012 T1)', 'Desocupados - mulheres (2012 T1)']
['Sigla', 'Código', 'Estado', 'Desocupados - homens (2026 T1)', 'Desocupados - mulheres (2026 T1)']
27 27


In [2]:
t2012 = t2012.rename(columns={
    "Desocupados - homens (2012 T1)": "homens_2012",
    "Desocupados - mulheres (2012 T1)": "mulheres_2012",
})
t2026 = t2026.rename(columns={
    "Desocupados - homens (2026 T1)": "homens_2026",
    "Desocupados - mulheres (2026 T1)": "mulheres_2026",
})

for quadro in (t2012, t2026):
    quadro["Código"] = quadro["Código"].astype(int)
    quadro["Sigla"] = quadro["Sigla"].astype(str).str.strip()
    quadro["Estado"] = quadro["Estado"].astype(str).str.strip()

print("chaves duplicadas em 2012:", int(t2012.duplicated(["Sigla", "Código", "Estado"]).sum()))
print("chaves duplicadas em 2026:", int(t2026.duplicated(["Sigla", "Código", "Estado"]).sum()))
print("soma homem+mulher 2012 fora de 100:", int(((t2012["homens_2012"] + t2012["mulheres_2012"]).round(1) != 100).sum()))
print("soma homem+mulher 2026 fora de 100:", int(((t2026["homens_2026"] + t2026["mulheres_2026"]).round(1) != 100).sum()))

comp = t2012.merge(
    t2026,
    on=["Sigla", "Código", "Estado"],
    how="outer",
    indicator=True,
    validate="one_to_one",
)
print(comp["_merge"].value_counts().to_dict())
comp = comp[comp["_merge"] == "both"].drop(columns="_merge")
print("linhas depois do merge:", len(comp))
comp.head()


chaves duplicadas em 2012: 0
chaves duplicadas em 2026: 0
soma homem+mulher 2012 fora de 100: 0
soma homem+mulher 2026 fora de 100: 0
{'both': 27, 'left_only': 0, 'right_only': 0}
linhas depois do merge: 27


,Sigla,Código,Estado,homens_2012,mulheres_2012,homens_2026,mulheres_2026
0,AC,12,Acre,45.3,54.7,53.2,46.8
1,AL,27,Alagoas,44.7,55.3,50.7,49.3
2,AM,13,Amazonas,47.5,52.5,46.8,53.2
3,AP,16,Amapá,48.1,51.9,51.1,48.9
4,BA,29,Bahia,43.8,56.2,44.7,55.3


In [3]:
bruto = ler_xls(DADOS / "Tabela 1.1.1.xls", "2022")
indicador_1 = bruto.iloc[8:41].copy()
indicador_1.columns = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
indicador_1 = indicador_1.reset_index(drop=True)
for coluna in indicador_1.columns[1:]:
    indicador_1[coluna] = pd.to_numeric(indicador_1[coluna], errors="coerce")

regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
ufs = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + regioes)].copy()
print("ufs na tabela de horas:", len(ufs))

cruzada = comp.merge(
    ufs,
    left_on="Estado",
    right_on="uf_regiao",
    how="left",
    validate="one_to_one",
)
print("estados sem hora correspondente:", int(cruzada["uf_regiao"].isna().sum()))
print("estados da tabela de horas que ficaram de fora:", sorted(set(ufs["uf_regiao"]) - set(comp["Estado"])))


ufs na tabela de horas: 27
estados sem hora correspondente: 0
estados da tabela de horas que ficaram de fora: []


In [4]:
tabela_cruzada = cruzada[
    [
        "Sigla",
        "Código",
        "Estado",
        "mulheres_2012",
        "mulheres_2026",
        "total",
        "mulher_branca",
        "mulher_preta_parda",
    ]
].sort_values("mulheres_2026", ascending=False)
tabela_cruzada.to_csv(SAIDAS / "desocupacao_e_horas.csv", sep=";", index=False)
tabela_lida = pd.read_csv(SAIDAS / "desocupacao_e_horas.csv", sep=";")
print(len(tabela_lida), tabela_lida.columns.tolist())
tabela_cruzada


27 ['Sigla', 'Código', 'Estado', 'mulheres_2012', 'mulheres_2026', 'total', 'mulher_branca', 'mulher_preta_parda']


,Sigla,Código,Estado,mulheres_2012,mulheres_2026,total,mulher_branca,mulher_preta_parda
20,RO,11,Rondônia,56.6,67.0,17.014162,20.412887,21.061410
24,SE,28,Sergipe,53.5,59.7,18.826541,21.740902,25.637681
12,MT,51,Mato Grosso,55.9,57.2,14.239989,17.782825,18.695923
21,RR,14,Roraima,58.2,56.6,13.639257,16.135877,16.560244
23,SC,42,Santa Catarina,57.3,55.4,16.039355,19.414764,19.646573
4,BA,29,Bahia,56.2,55.3,17.849776,22.572730,23.217119
11,MS,50,Mato Grosso do Sul,52.8,54.7,15.309783,18.593200,19.542761
18,RJ,33,Rio de Janeiro,55.8,54.6,16.716263,19.203741,20.957728
25,SP,35,São Paulo,54.6,53.8,17.071295,20.771186,21.767620
26,TO,17,Tocantins,44.7,53.8,15.815781,19.663713,19.629399


In [5]:
ordem = tabela_cruzada.sort_values("mulheres_2026", ascending=False)["Estado"]
longo = comp.melt(
    id_vars=["Sigla", "Código", "Estado"],
    value_vars=["mulheres_2012", "mulheres_2026"],
    var_name="Ano",
    value_name="Participacao_mulheres",
)
longo["Ano"] = longo["Ano"].map({
    "mulheres_2012": "2012 T1",
    "mulheres_2026": "2026 T1",
})

fig, ax = plt.subplots(figsize=(10, 10))
sns.barplot(
    data=longo,
    y="Estado",
    x="Participacao_mulheres",
    hue="Ano",
    order=ordem,
    ax=ax,
)
ax.axvline(50, color="firebrick", linestyle="--", linewidth=1)
ax.set_xlabel("Participação das mulheres entre as pessoas desocupadas (%)")
ax.set_ylabel("")
ax.set_title("Desocupação: participação feminina em 2012 T1 e 2026 T1")
ax.legend(title="Ano")
fig.tight_layout()
fig.savefig(SAIDAS / "participacao_feminina_desocupacao.png", dpi=120, bbox_inches="tight")
plt.show()

C:\Users\alzat\AppData\Local\Temp\ipykernel_7356\2614658494.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
